# Sound event detection — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def hz_to_mel(f): return 2595*np.log10(1+np.asarray(f)/700)
def mel_to_hz(m): return 700*(10**(np.asarray(m)/2595)-1)
def stft_mag(x,N=256,H=64):
    w=np.hanning(N); frames=[]
    for s in range(0,len(x)-N+1,H): frames.append(np.abs(np.fft.rfft(x[s:s+N]*w)))
    return np.array(frames).T
print('setup ok')

## Framewise multilabel event probabilities
Sound event detection thresholds frame probabilities into event timelines. These examples show BCE, threshold counts, multilabel overlap, smoothing, and a synthetic event spectrogram.

In [ ]:
# 1) binary cross-entropy on frame probabilities
p=np.array([0.1,0.7,0.8,0.2,0.6]); y=np.array([0,1,1,0,1]); bce=-np.mean(y*np.log(p)+(1-y)*np.log(1-p))
plt.figure(figsize=(6,3)); plt.plot(p,'-o',label='prob'); plt.plot(y,'s',label='label'); plt.legend(); plt.title(f'BCE={bce:.3f}')
assert abs(bce-0.284)<0.001

In [ ]:
# 2) thresholding gives TP/FP/FN counts
p=np.array([0.1,0.7,0.8,0.2,0.6]); y=np.array([0,1,1,0,1]).astype(bool); pred=p>=0.5; tp=np.sum(pred & y); fp=np.sum(pred & ~y); fn=np.sum(~pred & y)
plt.figure(figsize=(6,3)); plt.bar(['TP','FP','FN'],[tp,fp,fn]); plt.title('perfect toy detection at threshold 0.5')
assert (int(tp),int(fp),int(fn))==(3,0,0)

In [ ]:
# 3) multilabel frames allow overlapping events
Y=np.array([[1,0,0],[1,1,0],[0,1,0],[0,1,1],[0,0,1]]); probs=0.1+0.8*Y
plt.figure(figsize=(6,3)); plt.imshow(probs.T,aspect='auto',origin='lower'); plt.xlabel('frame'); plt.ylabel('event'); plt.title('multiple events can be active')
assert Y[1].sum()==2 and Y[3].sum()==2

In [ ]:
# 4) smoothing removes flicker before segmenting
p=np.array([0.1,0.8,0.4,0.9,0.2,0.1]); smooth=np.convolve(p,np.ones(3)/3,mode='same'); raw=(p>=0.5); sm=(smooth>=0.4)
plt.figure(figsize=(6,3)); plt.plot(p,'-o',label='raw prob'); plt.plot(smooth,'-s',label='smoothed'); plt.legend(); plt.title('smoothing changes event continuity')
assert raw.sum()==2 and sm.sum()==3

In [ ]:
# 5) synthetic event spectrogram has a localized high-energy patch
S=np.zeros((32,40)); S[20:26,12:22]=1.0; S+=0.05*np.random.default_rng(2).random(S.shape); frame_energy=S[20:26].mean(0); active=frame_energy>0.5
plt.figure(figsize=(6,3)); plt.imshow(S,origin='lower',aspect='auto'); plt.plot(np.arange(40),active*30,c='w'); plt.title('event localized in time and frequency')
assert active.sum()==10